# 【notebook｜N-02】Node 类练习：显式栈迭代、parent 指针、两层 AND

> 前置：`examples/update_modes_walkthrough.ipynb`（N-01，Node 与递归基础）；本 notebook 自包含、只用标准库。
> 规范：`examples/00-风格与编号规范.md`（DOC-EX-STYLE）｜文件级 Tag：N-02。

**三道练习**

1. 把递归版 `compute_value_target` 改写成**显式栈的迭代版**，对照两种写法；
2. 给 `Node` 加 **parent 指针**，实现 `path_to_root(node)`；
3. 建一棵**两层 AND** 的树，先手算 `value_target` 表格，再跑断言验证。

## §0 热身：Node 类与一棵简单树

**【N-02.m2】**


先把 N-01 的 `Node` 原样搬过来（方便对照），再用它建一棵熟悉的小树当基准。

In [1]:
# 【N-02.c1】EX-0.1｜Node 类（与 N-01 相同）＋简单树工厂
class Node:
    def __init__(self, name, to_play="OR", terminal=False):
        self.name = name
        self.to_play = to_play
        self.terminal = terminal
        self.children = {}
        self.is_optimal = False
        self.value_target = None

    def add(self, action, child):
        self.children[action] = child


def build_simple_tree():
    """N-01 用过的 7 节点树：root(OR) -> split(AND) -> [x 分支, y 分支]。"""
    leaf_x = Node("x: proved", terminal=True)
    node_x = Node("x-branch(OR)"); node_x.add("close-x", leaf_x)
    leaf_y = Node("y: proved", terminal=True)
    node_y2 = Node("y-mid(OR)"); node_y2.add("close-y", leaf_y)
    node_y = Node("y-branch(OR)"); node_y.add("progress-y", node_y2)
    and_node = Node("split(AND)", to_play="AND")
    and_node.add("focus 0", node_x); and_node.add("focus 1", node_y)
    root = Node("root(OR)"); root.add("split", and_node)
    for n in (root, and_node, node_x, node_y, node_y2, leaf_x, leaf_y):
        n.is_optimal = True
    return root, {"root": root, "and": and_node, "x": node_x, "y": node_y}


root0, nodes0 = build_simple_tree()
print("节点数:", 1 + len(root0.children["split"].children) + 4)

节点数: 7


### 0.1 递归版基准

`compute_value_target` 是**后序**：子节点算完，才能算自己。
OR 只沿 `is_optimal` 的那个子走；AND 要算**全部子**再取 min——这两条规则后面迭代版必须一模一样。

In [2]:
# 【N-02.c2】EX-0.2｜递归版（复习）：先跑一遍当基准
def compute_value_target(node):
    if node.terminal:
        node.value_target = 0.0
        return 0.0
    if node.to_play == "OR":
        action, child = next((a, c) for a, c in node.children.items() if c.is_optimal)
        value = -1.0 + compute_value_target(child)
    else:
        value = min(compute_value_target(c) for c in node.children.values())
    node.value_target = value
    return value


root1, nodes1 = build_simple_tree()
recursive_value = compute_value_target(root1)
print("递归结果：")
for name in ("root", "and", "x", "y"):
    n = nodes1[name]
    print(f"  {name:<5} value_target={n.value_target:+.1f}  d={abs(float(n.value_target)):.1f}")
assert recursive_value == -3.0

递归结果：
  root  value_target=-3.0  d=3.0
  and   value_target=-2.0  d=2.0
  x     value_target=-1.0  d=1.0
  y     value_target=-2.0  d=2.0


## §1 练习①：把递归改写成显式栈

**【N-02.m4】**


递归是让 Python 的调用栈替你记"回来的路"；迭代版就是自己用 `list` 当栈。
后序计算的标准写法是**两阶段**：

```text
栈里放 (节点, children_done)：
  第一次弹出（False）：把 (自己, True) 压回去，再把待处理的子节点压进去；
  第二次弹出（True） ：此时子节点都算完了，按 OR/AND 规则写回 value_target。
```

规则照抄：**OR 只压最优子；AND 压全部子**。

In [3]:
# 【N-02.c3】EX-1.1｜迭代版：显式栈 + 两阶段后序
def compute_value_target_iter(root):
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if not done:
            if node.terminal:
                node.value_target = 0.0
                continue
            stack.append((node, True))                  # 先记下"回来还要算自己"
            if node.to_play == "OR":
                child = next(c for c in node.children.values() if c.is_optimal)
                stack.append((child, False))            # OR：只处理最优子
            else:
                for child in node.children.values():    # AND：全部子
                    stack.append((child, False))
        else:
            if node.to_play == "OR":
                child = next(c for c in node.children.values() if c.is_optimal)
                node.value_target = -1.0 + child.value_target
            else:
                node.value_target = min(c.value_target for c in node.children.values())
    return root.value_target


root2, nodes2 = build_simple_tree()
iter_value = compute_value_target_iter(root2)
print("迭代结果:", f"{iter_value:+.1f}")
assert iter_value == -3.0
for key in ("root", "and", "x", "y"):
    assert nodes2[key].value_target == nodes1[key].value_target, key
print("迭代版与递归版逐节点一致 ✓")

迭代结果: -3.0
迭代版与递归版逐节点一致 ✓


In [4]:
# 【N-02.c4】EX-1.2｜访问顺序对照：LIFO 的坑与修法
def iter_visit_order(root):
    stack = [(root, False)]
    order = []
    while stack:
        node, done = stack.pop()
        if not done:
            if node.terminal:
                node.value_target = 0.0
                order.append(node.name)
                continue
            stack.append((node, True))
            if node.to_play == "OR":
                stack.append((next(c for c in node.children.values() if c.is_optimal), False))
            else:
                # 栈是后进先出：反着压，弹出顺序才和递归的“从左到右”一致
                for child in reversed(list(node.children.values())):
                    stack.append((child, False))
        else:
            if node.to_play == "AND":
                node.value_target = min(c.value_target for c in node.children.values())
            else:
                node.value_target = -1.0 + next(
                    c for c in node.children.values() if c.is_optimal).value_target
            order.append(node.name)
    return order


def rec_visit_order(node, order):
    if node.terminal:
        node.value_target = 0.0
        order.append(node.name)
        return 0.0
    if node.to_play == "OR":
        child = next(c for c in node.children.values() if c.is_optimal)
        value = -1.0 + rec_visit_order(child, order)
    else:
        value = min(rec_visit_order(c, order) for c in node.children.values())
    node.value_target = value
    order.append(node.name)
    return value


root3, _ = build_simple_tree()
rec_order = []
rec_visit_order(root3, rec_order)
root4, _ = build_simple_tree()
iter_order = iter_visit_order(root4)
print("递归访问顺序:", rec_order)
print("迭代访问顺序:", iter_order)
assert rec_order == iter_order
print()
print("小结：值一样、访问顺序也能完全一致；差别在于——")
print("  递归：行数少，但深树可能爆栈；")
print("  迭代：显式管理“回来时干什么”，不怕深树，还能随时打印/中断栈。")

递归访问顺序: ['x: proved', 'x-branch(OR)', 'y: proved', 'y-mid(OR)', 'y-branch(OR)', 'split(AND)', 'root(OR)']
迭代访问顺序: ['x: proved', 'x-branch(OR)', 'y: proved', 'y-mid(OR)', 'y-branch(OR)', 'split(AND)', 'root(OR)']

小结：值一样、访问顺序也能完全一致；差别在于——
  递归：行数少，但深树可能爆栈；
  迭代：显式管理“回来时干什么”，不怕深树，还能随时打印/中断栈。


## §2 练习②：parent 指针与 `path_to_root`

**【N-02.m5】**


`Node` 一直只有向下的 `children`；要回答"这个节点是从哪条路来的"，就得有向上的 parent 指针。两种做法：

- **方案 A（不改类）**：写一次遍历 `attach_parents(root)`，给每个子节点补上 `parent` / `parent_action`；
- **方案 B（增量维护）**：子类化 `PNode`，在 `add()` 里顺手把双向连接建好。

拿到 parent 后，`path_to_root` 就是一个 `while` 循环。

In [5]:
# 【N-02.c5】EX-2.1｜方案 A：一次性补挂 parent + path_to_root
def attach_parents(node, parent=None, action=None):
    node.parent = parent
    node.parent_action = action
    for act, child in node.children.items():
        attach_parents(child, node, act)


def path_to_root(node):
    actions = []
    cur = node
    while cur is not None:
        actions.append(cur.parent_action)
        cur = getattr(cur, "parent", None)
    actions.pop()                       # 根的 parent_action 是 None，去掉
    return list(reversed(actions))


root5, nodes5 = build_simple_tree()
attach_parents(root5)
leaf = nodes5["y"].children["progress-y"].children["close-y"]
print("目标叶子:", leaf.name)
print("path_to_root:", path_to_root(leaf))
assert path_to_root(leaf) == ["split", "focus 1", "progress-y", "close-y"]
print("root 的 parent:", getattr(root5, "parent", None))
assert path_to_root(root5) == []
print("根节点回溯 → 空列表 ✓")

目标叶子: y: proved
path_to_root: ['split', 'focus 1', 'progress-y', 'close-y']
root 的 parent: None
根节点回溯 → 空列表 ✓


In [6]:
# 【N-02.c6】EX-2.2｜方案 B：子类 PNode，在 add() 里维护双向连接
class PNode(Node):
    def __init__(self, name, to_play="OR", terminal=False):
        super().__init__(name, to_play, terminal)   # 先让父类完成初始化
        self.parent = None
        self.parent_action = None

    def add(self, action, child):
        super().add(action, child)                  # 先放孩子
        child.parent = self                         # 再双向连接
        child.parent_action = action


r = PNode("r"); m = PNode("m"); t = PNode("t", terminal=True)
r.add("a", m); m.add("b", t)
print("m.parent is r     :", m.parent is r)
print("t.parent_action   :", t.parent_action)
print("t 的 path_to_root :", path_to_root(t))
assert path_to_root(t) == ["a", "b"]

extra = PNode("extra"); m.add("c", extra)          # 新分支自动带 parent，无需重跑遍历
assert path_to_root(extra) == ["a", "c"]
print("新增分支后 path_to_root(extra):", path_to_root(extra))

# 注意：attach_parents 会给普通 Node 动态加属性；PNode 则在类里显式声明。
# 两种都能 work，但“同一个子节点挂两个父亲”（DAG）时单 parent 会互相覆盖——见 §4 思考题。

m.parent is r     : True
t.parent_action   : b
t 的 path_to_root : ['a', 'b']
新增分支后 path_to_root(extra): ['a', 'c']


## §3 练习③：两层 AND 的树——先手算，再跑断言

**【N-02.m6】**


目标树（**AND 套 AND**）：

```text
root(OR) ─"split"─> split1(AND)
                     ├─ "focus 0" ─> a(OR)  ─"close-a"─> a:proved
                     └─ "focus 1" ─> split2(AND)
                                      ├─ "focus 0" ─> b1(OR) ─"close-b1"─> b1:proved
                                      └─ "focus 1" ─> b2(OR) ─"mid-b2"─> b2m(OR) ─"close-b2"─> b2:proved
```

**先手算**（覆盖右侧一列再对答案）：

| 节点 | 规则 | value_target |
| --- | --- | --- |
| a / b1 / b2 叶子 | 终端 | 0 |
| b2m | OR：-1 + 子 | -1 |
| b2 | OR：-1 + b2m | -2 |
| b1 | OR：-1 + 子 | -1 |
| split2 | AND：min(b1, b2) | **-2** |
| a | OR：-1 + 子 | -1 |
| split1 | AND：min(a, split2) | **-2** |
| root | OR：-1 + split1 | **-3** |

关键观察：**两次 min 都被“更深的那条分支”选中**——AND 取最坏（最长）分支决定整体。

In [7]:
# 【N-02.c7】EX-3.1｜建两层 AND 树；递归计算；逐项断言（对照手算表）
def build_two_and_tree():
    leaf_a = Node("a: proved", terminal=True); na = Node("a(OR)"); na.add("close-a", leaf_a)
    leaf_b1 = Node("b1: proved", terminal=True); b1 = Node("b1(OR)"); b1.add("close-b1", leaf_b1)
    leaf_b2 = Node("b2: proved", terminal=True); b2m = Node("b2m(OR)"); b2m.add("close-b2", leaf_b2)
    b2 = Node("b2(OR)"); b2.add("mid-b2", b2m)
    split2 = Node("split2(AND)", to_play="AND"); split2.add("focus 0", b1); split2.add("focus 1", b2)
    split1 = Node("split1(AND)", to_play="AND"); split1.add("focus 0", na); split1.add("focus 1", split2)
    root = Node("root(OR)"); root.add("split", split1)
    for n in (root, split1, split2, na, b1, b2, b2m, leaf_a, leaf_b1, leaf_b2):
        n.is_optimal = True
    return root, {"root": root, "split1": split1, "split2": split2,
                  "a": na, "b1": b1, "b2": b2, "b2m": b2m}


root6, t1 = build_two_and_tree()
value = compute_value_target(root6)
hand = {"a": -1.0, "b1": -1.0, "b2m": -1.0, "b2": -2.0,
        "split2": -2.0, "split1": -2.0, "root": -3.0}
print("逐项对照手算表：")
for key, expected in hand.items():
    got = t1[key].value_target
    status = "OK" if got == expected else "MISMATCH"
    print(f"  {key:<7} 手算 {expected:+.1f}  实算 {got:+.1f}  {status}")
    assert got == expected, key
assert value == -3.0
print()
print("两层 min 都被更深分支选中：split2 = min(-1, -2) = -2，split1 = min(-1, -2) = -2 ✓")

逐项对照手算表：
  a       手算 -1.0  实算 -1.0  OK
  b1      手算 -1.0  实算 -1.0  OK
  b2m     手算 -1.0  实算 -1.0  OK
  b2      手算 -2.0  实算 -2.0  OK
  split2  手算 -2.0  实算 -2.0  OK
  split1  手算 -2.0  实算 -2.0  OK
  root    手算 -3.0  实算 -3.0  OK

两层 min 都被更深分支选中：split2 = min(-1, -2) = -2，split1 = min(-1, -2) = -2 ✓


In [8]:
# 【N-02.c8】EX-3.2｜同一棵树上：迭代版 + parent 指针
root7, t2 = build_two_and_tree()
assert compute_value_target_iter(root7) == -3.0
for key, node in t2.items():
    assert node.value_target == t1[key].value_target, key
print("迭代版与递归版在两层 AND 树上逐节点一致 ✓")

attach_parents(root7)
deep = t2["b2m"].children["close-b2"]          # "b2: proved" 叶子
print("最深叶子的 path_to_root:", path_to_root(deep))
assert path_to_root(deep) == ["split", "focus 1", "focus 1", "mid-b2", "close-b2"]
print("✓ 两层 focus 的路径也能完整回溯")

迭代版与递归版在两层 AND 树上逐节点一致 ✓
最深叶子的 path_to_root: ['split', 'focus 1', 'focus 1', 'mid-b2', 'close-b2']
✓ 两层 focus 的路径也能完整回溯


## §4 小结与再练习

**【N-02.m7】**


- **递归 vs 迭代**：递归让语言压栈；迭代自己用 `list` 当栈（两阶段后序）。规则必须一致：
  **OR 只沿最优子算，AND 算全部子取 min**；
- **parent 指针**：方案 A 一次遍历补挂；方案 B（子类 `PNode`）在 `add()` 里维护。`path_to_root` 就是一个 `while`；
- **两层 AND**：min 会连续两次选择更深的分支——这正是"最坏分支决定整体进度"的体现。

**思考题**

1. 给 `compute_value_target_iter` 加一个"返回访问顺序"的返回值，验证与递归顺序一致（提示：AND 的子反着压栈）；
2. 若同一个子节点被两个父亲引用（DAG），单 parent 指针会发生什么？应该怎么改（提示：parent 变成列表，或记录"最佳父亲"）；
3. 两层 AND 树里给 b2 再续一层（让 b2 变成 -3），手算 `root` 会变成多少？改代码验证。